[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARODBE/UAX_Fundamento_Ciencia_de_Datos/blob/main/Notebooks/Fase_6_Regresion_Logistica_DILI.ipynb)

# Capítulo 6. Regresión logística para la clase DILI

Uso la etiqueta DILI como variable binaria: 1 indica clasificación positiva y 0 negativa. De los descriptores de RDKit, incluyo MolLogP y selecciono en train los once restantes con mayor asociación de Spearman. LipinskiViolations se excluye por ser una variable derivada. Limito el número para mantener un modelo interpretable y reduzco redundancia con correlación por pares y VIF. Evalúo el rendimiento en test y comparo sus métricas con train.


In [1]:
# Instalo las librerías que usaré para la regresión y su gráfico de odds ratios.\n%pip -q install statsmodels forestplot

!pip install statsmodels forestplot

## 6.1. Lectura de los datos

In [2]:
# Cargo las librerías para leer y analizar las tablas.
from pathlib import Path  # Organizo las rutas locales.
import pandas as pd  # Leo y preparo tablas.
import numpy as np  # Calculo estadísticas numéricas.
from IPython.display import display  # Muestro resultados en Colab.

# Defino una carpeta local para guardar copias de trabajo.
DATA_DIR = Path('/content/datos_dili')  # Uso la carpeta de Colab.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta si hace falta.

# Indico la carpeta pública con los datos que preparé en la fase 2.
BASE_URL = 'https://raw.githubusercontent.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos/main/Data/'  # Defino el enlace base.
dilist_original = pd.read_csv(BASE_URL + 'DILIst_FDA.csv')  # Cargo la fuente original.
estructuras = pd.read_csv(BASE_URL + 'estructuras_nombres.csv')  # Cargo el mapa de nombres y estructuras.
datos_limpios = pd.read_csv(BASE_URL + 'dilist_descriptores.csv')  # Cargo los compuestos con todos los descriptores.
train_df = pd.read_csv(BASE_URL + 'dilist_train.csv')  # Cargo entrenamiento.
test_df = pd.read_csv(BASE_URL + 'dilist_test.csv')  # Cargo prueba.

# Creo una tabla completa para revisar cobertura y calidad de estructuras.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Renombro la clase.
datos = dilist.merge(estructuras, on='CompoundName', how='left')  # Uno las etiquetas con el estado de búsqueda.
claves_con_etiqueta = datos.dropna(subset=['InChIKey']).groupby('InChIKey')['DILI'].nunique()  # Cuento clases por estructura.
claves_conflictivas = claves_con_etiqueta[claves_con_etiqueta > 1].index  # Identifico etiquetas contradictorias.
datos_conflicto = datos[datos['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo los registros conflictivos.

# Identifico todos los descriptores numéricos salvo etiquetas e identificadores.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'CID', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito datos no moleculares.
DESCRIPTORES = [columna for columna in datos_limpios.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades moleculares.

# Guardo copias locales de las tablas de trabajo.
for nombre, tabla in [('dilist_descriptores.csv', datos_limpios), ('dilist_train.csv', train_df), ('dilist_test.csv', test_df)]:  # Recorro las tablas.
    tabla.to_csv(DATA_DIR / nombre, index=False)  # Guardo una copia en Colab.
print('He cargado los datos del repositorio:', DATA_DIR)  # Confirmo la ruta local.
print('Descriptores numéricos disponibles:', len(DESCRIPTORES))  # Muestro el catálogo disponible.

# Defino las cinco propiedades usadas para marcar observaciones extremas.
variables_filtro = ['MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']  # Uso las propiedades del EDA.
variables_filtro = [v for v in variables_filtro if v in train_df.columns]  # Compruebo que existan.
train_original = train_df.copy()  # Guardo train antes del filtro.
test_original = test_df.copy()  # Guardo test antes del filtro.

# Calculo los límites de 1,5 × RIC únicamente con train.
q1 = train_original[variables_filtro].quantile(0.25)  # Calculo el primer cuartil.
q3 = train_original[variables_filtro].quantile(0.75)  # Calculo el tercer cuartil.
ric = q3 - q1  # Calculo el rango intercuartílico.
limite_inferior = q1 - 1.5 * ric  # Defino los límites inferiores.
limite_superior = q3 + 1.5 * ric  # Defino los límites superiores.
fuera_train = (train_original[variables_filtro].lt(limite_inferior) | train_original[variables_filtro].gt(limite_superior)).any(axis=1)  # Marco extremos de train.
fuera_test = (test_original[variables_filtro].lt(limite_inferior) | test_original[variables_filtro].gt(limite_superior)).any(axis=1)  # Aplico los mismos límites a test.
train_df = train_original.loc[~fuera_train].reset_index(drop=True)  # Retiro extremos de train.
test_df = test_original.loc[~fuera_test].reset_index(drop=True)  # Retiro extremos de test.
print(f"Filtro 1,5 × RIC: train {len(train_df)}/{len(train_original)}; test {len(test_df)}/{len(test_original)} compuestos.")  # Informo los tamaños.
print('Clases en train:', train_df['DILI'].value_counts().sort_index().to_dict())  # Compruebo train.
print('Clases en test:', test_df['DILI'].value_counts().sort_index().to_dict())  # Compruebo test.
print('Límites calculados en train:', {v: (round(limite_inferior[v], 2), round(limite_superior[v], 2)) for v in variables_filtro})  # Muestro los umbrales.

He cargado los datos del repositorio: /content/datos_dili
Descriptores numéricos disponibles: 218
Filtro 1,5 × RIC: train 832/947; test 201/237 compuestos.
Clases en train: {0: 307, 1: 525}
Clases en test: {0: 68, 1: 133}
Límites calculados en train: {'MolWt': (np.float64(-1.01), np.float64(687.9)), 'MolLogP': (np.float64(-3.82), np.float64(8.22)), 'NumHDonors': (np.float64(-2.0), np.float64(6.0)), 'NumHAcceptors': (np.float64(-1.5), np.float64(10.5)), 'TPSA': (np.float64(-59.66), np.float64(220.06))}


## 6.2. Selección, colinealidad y ajuste

De los 217 descriptores calculados con RDKit se mantiene MolLogP como predictor de interés y se excluye LipinskiViolations, que es una variable derivada. Tras el filtro de extremos, siete descriptores no presentan variación en train y se descartan; quedan 210 candidatos. La selección incluye MolLogP y los once descriptores restantes con mayor asociación de Spearman con DILI en train. El filtro por correlación reduce la redundancia entre predictores. Después se revisa el VIF y se eliminan iterativamente los valores más altos, manteniendo MolLogP en el modelo. La imputación y el escalado se ajustan solo con train y se aplican a test.


In [ ]:
# Importo herramientas para preparar y ajustar la regresión logística.
from sklearn.impute import SimpleImputer  # Completo posibles valores ausentes.
from sklearn.preprocessing import StandardScaler  # Estandarizo los predictores.
import statsmodels.api as sm  # Ajusto el modelo y obtengo intervalos y valores p.

# Excluyo la variable derivada, la etiqueta y los identificadores; MolLogP se conserva.
columnas_excluir = ['LipinskiViolations', 'DILI', 'DILIST_ID']  # Mantengo LogP como predictor.
# Separo los descriptores RDKit de la variable derivada de Lipinski.
descriptores_rdkit = [variable for variable in DESCRIPTORES if variable != 'LipinskiViolations']  # Identifico los descriptores calculados con RDKit.

# Identifico las columnas que se excluyen por el planteamiento del modelo.
excluidos_modelo = [variable for variable in DESCRIPTORES if variable in columnas_excluir]  # Guardo solo las exclusiones presentes.
predictores_base = [variable for variable in DESCRIPTORES if variable not in columnas_excluir]  # Quito la respuesta, su derivación y los identificadores.

# Quito columnas constantes o sin valores observados usando solo entrenamiento.
sin_variacion = [variable for variable in predictores_base if train_df[variable].nunique(dropna=True) <= 1]  # Detecto columnas sin variabilidad.
predictores = [variable for variable in predictores_base if train_df[variable].nunique(dropna=True) > 1]  # Conservo predictores variables.

# Muestro el desglose para que se vea el paso de descriptores a predictores.
print('Descriptores RDKit disponibles:', len(descriptores_rdkit))  # Informo cuántos calcula RDKit.
print('Columnas excluidas por el modelo:', len(excluidos_modelo), excluidos_modelo)  # Explico las exclusiones definidas.
print('Descriptores sin variación en train:', len(sin_variacion), sin_variacion)  # Muestro columnas descartadas por falta de variación.
print('Descriptores candidatos:', len(predictores))  # Informo el total final de candidatos.

# Reviso las asociaciones bivariadas con DILI usando solo entrenamiento.
asociacion = train_df[predictores + ['DILI']].corr(method='spearman')['DILI'].drop('DILI').dropna().abs().sort_values(ascending=False)  # Ordeno la asociación absoluta.
# Incluyo MolLogP y completo doce candidatos con los once descriptores más asociados.
otros_candidatos = asociacion.drop(index='MolLogP', errors='ignore').head(11).index.tolist()  # Selecciono once variables adicionales.
candidatos = ['MolLogP'] + otros_candidatos  # Mantengo LogP entre los candidatos.
print('Candidatos con mayor asociación a DILI:', len(candidatos))  # Muestro el número seleccionado.

# Quito variables redundantes entre los candidatos.
correlacion = train_df[candidatos].corr(method='spearman').abs()  # Calculo correlación absoluta.
seleccionados = []  # Creo la lista de predictores finales.
for variable in candidatos:  # Reviso candidatos en orden de asociación.
    if not any(correlacion.loc[variable, otra] >= 0.90 for otra in seleccionados):  # Compruebo redundancia.
        seleccionados.append(variable)  # Conservo variables por debajo del umbral.
print('Descriptores finales tras quitar redundancia:', seleccionados)  # Informo el conjunto final.

# Muestro el criterio de selección y el tipo de propiedad de cada descriptor.
descripcion = {'MolLogP': 'estimación de lipofilia calculada por RDKit', 'FractionCSP3': 'proporción de carbonos sp3', 'BCUT2D_LOGPLOW': 'topología ponderada por contribuciones de LogP', 'fr_Al_OH': 'recuento de grupos hidroxilo alifáticos', 'SMR_VSA6': 'superficie agrupada por refractividad molar', 'SlogP_VSA2': 'superficie agrupada por contribuciones de LogP', 'fr_Ndealkylation1': 'motivo estructural de N-desalquilación', 'BCUT2D_MRHI': 'topología ponderada por refractividad molar', 'fr_quatN': 'recuento de nitrógenos cuaternarios', 'PEOE_VSA1': 'superficie agrupada por carga parcial', 'NumAtomStereoCenters': 'número de centros estereogénicos', 'VSA_EState9': 'superficie agrupada por estado electrotopológico', 'BCUT2D_CHGLO': 'topología ponderada por carga parcial'}  # Describo los predictores seleccionados.
tabla_seleccion = pd.DataFrame({'descriptor': seleccionados, 'rho_Spearman_train': train_df[seleccionados + ['DILI']].corr(method='spearman')['DILI'].loc[seleccionados].values, 'tipo de característica': [descripcion.get(variable, 'descriptor molecular RDKit') for variable in seleccionados]})  # Uno su significado y asociación.
display(tabla_seleccion.round(3))  # Muestro por qué quedaron en el modelo.
print('Los selecciono por su asociación bivariada más alta en train; después retiro redundancia entre predictores. Son candidatos exploratorios, no causas ni una lista universal de hepatotoxicidad.')  # Justifico la elección.


# Preparo la variable objetivo y ajusto transformaciones usando solo train.
y_train = train_df['DILI'].astype(int)  # Defino las etiquetas de entrenamiento.
y_test = test_df['DILI'].astype(int)  # Defino las etiquetas de prueba.
imputador = SimpleImputer(strategy='median')  # Creo imputación por mediana.
imputador.fit(train_df[seleccionados])  # Aprendo medianas solo en train.
X_train_imp = imputador.transform(train_df[seleccionados])  # Transformo train.
X_test_imp = imputador.transform(test_df[seleccionados])  # Aplico medianas de train a test.
escalador = StandardScaler()  # Creo el escalador.
escalador.fit(X_train_imp)  # Aprendo medias y desviaciones solo con train.
X_train = escalador.transform(X_train_imp)  # Estandarizo train.
X_test = escalador.transform(X_test_imp)  # Transformo test con el mismo ajuste.

# Organizo las matrices con nombres para interpretar cada coeficiente.
X_train_df = pd.DataFrame(X_train, columns=seleccionados, index=train_df.index)  # Etiqueto las columnas de train.
X_test_df = pd.DataFrame(X_test, columns=seleccionados, index=test_df.index)  # Etiqueto las columnas de test.
X_train_sm = sm.add_constant(X_train_df, has_constant='add')  # Añado el intercepto.
X_test_sm = sm.add_constant(X_test_df, has_constant='add')  # Añado el mismo intercepto a test.

# Reviso colinealidad multivariante en los predictores estandarizados de train.
from statsmodels.stats.outliers_influence import variance_inflation_factor  # Calculo VIF.
variables_retiradas_vif = []  # Guardo los predictores eliminados por redundancia.
while X_train_df.shape[1] > 2:  # Mantengo al menos dos variables explicativas.
    tabla_vif = pd.DataFrame({'descriptor': X_train_df.columns, 'VIF': [variance_inflation_factor(X_train_df.values, i) for i in range(X_train_df.shape[1])]})  # Calculo el VIF de cada columna.
    if tabla_vif['VIF'].max() <= 5:  # Uso cinco como umbral de revisión.
        break  # Paro cuando los valores quedan bajo el umbral.
    tabla_retirables = tabla_vif[tabla_vif['descriptor'] != 'MolLogP']  # Protejo MolLogP como predictor predefinido.
    if tabla_retirables.empty:  # Compruebo que quede otra variable que pueda retirar.
        break  # Mantengo el modelo si solo queda MolLogP.
    fila_maxima = tabla_retirables['VIF'].idxmax()  # Localizo el VIF más alto entre las demás variables.
    variable_retirada = tabla_vif.loc[fila_maxima, 'descriptor']  # Identifico el predictor redundante.
    variables_retiradas_vif.append((variable_retirada, tabla_vif.loc[fila_maxima, 'VIF']))  # Registro el predictor y su VIF.
    X_train_df = X_train_df.drop(columns=variable_retirada)  # Quito la variable de train.
    X_test_df = X_test_df.drop(columns=variable_retirada)  # Aplico la misma eliminación a test.
    seleccionados.remove(variable_retirada)  # Actualizo la lista del modelo.
tabla_vif = pd.DataFrame({'descriptor': X_train_df.columns, 'VIF': [variance_inflation_factor(X_train_df.values, i) for i in range(X_train_df.shape[1])]})  # Recalculo VIF finales.
display(tabla_vif.sort_values('VIF', ascending=False).round(2))  # Muestro los predictores retenidos.
print('Predictores retirados por VIF:', [(nombre, round(float(vif), 2)) for nombre, vif in variables_retiradas_vif] if variables_retiradas_vif else 'ninguno')  # Explico qué cambió.
print('El umbral VIF de 5 guía la reducción de colinealidad; MolLogP se conserva por el objetivo del análisis.')  # Explico el criterio.
X_train_sm = sm.add_constant(X_train_df, has_constant='add')  # Actualizo train tras el filtro.
X_test_sm = sm.add_constant(X_test_df, has_constant='add')  # Actualizo test con las mismas columnas.

# Pondero las clases de train para compensar su distinta frecuencia.
pesos = np.where(y_train == 1, len(y_train) / (2 * (y_train == 1).sum()), len(y_train) / (2 * (y_train == 0).sum()))  # Calculo pesos equilibrados.
modelo = sm.GLM(y_train, X_train_sm, family=sm.families.Binomial(), freq_weights=pesos).fit()  # Ajusto regresión logística ponderada.


## 6.3. Odds ratios e intervalos de confianza

Represento con forestplot los odds ratios, sus intervalos de confianza del 95 % y los valores p. Como los predictores están estandarizados, cada OR corresponde a un aumento de una desviación estándar, manteniendo constantes los demás predictores. La línea vertical marca OR = 1, valor de referencia sin cambio en las odds.


In [ ]:
# Importo las herramientas para representar e interpretar odds ratios.
import matplotlib.pyplot as plt  # Dibujo el forest plot.
import forestplot as fp  # Uso la librería solicitada para el forest plot.

# Calculo odds ratios e intervalos de confianza del 95 %.
intervalos = modelo.conf_int().loc[seleccionados]  # Extraigo intervalos de los predictores retenidos.
odds = pd.DataFrame({'descriptor': seleccionados, 'odds_ratio': np.exp(modelo.params.loc[seleccionados].values), 'ic_inferior': np.exp(intervalos[0].values), 'ic_superior': np.exp(intervalos[1].values), 'p_value': modelo.pvalues.loc[seleccionados].values})  # Organizo estimaciones e incertidumbre.
odds['p_text'] = odds['p_value'].apply(lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')  # Presento p pequeños sin redondearlos a cero.

# Muestro una figura con estimaciones, intervalos y valores p.
figura, eje = plt.subplots(figsize=(12, max(5, 0.42 * len(odds))))  # Ajusto el tamaño al número de predictores.
fp.forestplot(dataframe=odds, estimate='odds_ratio', ll='ic_inferior', hl='ic_superior', varlabel='descriptor', pval=None, rightannote=['p_text'], right_annoteheaders=['p'], logscale=True, sort=True, xlabel='Odds ratio e IC del 95 % (escala logarítmica)', xticks=[0.1, 0.5, 1, 2, 10], decimal_precision=2, color_alt_rows=True, xline=1, xlinestyle='--', table=True, ylabel=' ', ax=eje)  # Dibujo el forest plot sin marcas superpuestas.
plt.tight_layout()  # Ajusto los nombres y el eje.
plt.show()  # Presento los odds ratios.

# Comento los dos odds ratios extremos calculados.
orden_odds = odds.sort_values('odds_ratio')  # Ordeno los OR.
for fila in [orden_odds.iloc[0], orden_odds.iloc[-1]]:  # Reviso el OR menor y el mayor.
    cambio = 100 * (fila['odds_ratio'] - 1)  # Calculo el cambio porcentual en odds.
    evidencia = 'el IC queda a un lado de 1' if fila['ic_superior'] < 1 or fila['ic_inferior'] > 1 else 'el IC incluye 1'  # Valoro si el intervalo cruza la referencia.
    print(f"{fila['descriptor']}: OR = {fila['odds_ratio']:.2f}, cambio estimado de odds {cambio:+.1f}% por una desviación estándar; {evidencia}.")  # Interpreto el resultado mostrado.
print('Estos son efectos ajustados dentro de este modelo y no implican causalidad; la selección previa hace que los valores p sean exploratorios.')  # Aclaro el alcance inferencial.


## Lectura de los odds ratios

El forest plot presenta los odds ratios ajustados y sus intervalos de confianza del 95 %. MolLogP permanece en el modelo por su interés fisicoquímico. Para cada predictor estandarizado, un OR superior a 1 indica mayores odds de clasificación DILI positiva por un aumento de una desviación estándar; un OR inferior a 1 indica menores odds. Si el intervalo incluye 1, la estimación no muestra evidencia clara de asociación al nivel habitual de significación. La selección se realizó en train, por lo que estos resultados se consideran exploratorios y no causales.

## 6.4. Rendimiento de clasificación en train y test

Comparo accuracy, precision, sensibilidad (recall), especificidad, F1 y AUC-ROC entre las dos particiones. En test, muestro también la matriz de confusión y la curva ROC en una misma figura para relacionar los indicadores con los errores observados.


In [ ]:
# Importo las métricas necesarias para comparar clasificación y curvas ROC.
import matplotlib.pyplot as plt  # Dibujo la matriz y la curva ROC en una figura.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve  # Calculo evaluación binaria.

# Estimo probabilidades en las dos particiones con el modelo ya ajustado.
prob_train = modelo.predict(X_train_sm)  # Predigo probabilidades en train.
prob_test = modelo.predict(X_test_sm)  # Predigo probabilidades en test.
pred_train = (prob_train >= 0.5).astype(int)  # Clasifico train con umbral 0,5.
pred_test = (prob_test >= 0.5).astype(int)  # Clasifico test con el mismo umbral.

# Calculo especificidad además de las métricas de clasificación solicitadas.
def calcular_metricas(y_real, y_predicho, probabilidad):  # Defino el resumen para una partición.
    tn, fp, fn, tp = confusion_matrix(y_real, y_predicho, labels=[0, 1]).ravel()  # Extraigo verdaderos y falsos positivos y negativos.
    especificidad = tn / (tn + fp) if (tn + fp) else 0  # Calculo la tasa de verdaderos negativos.
    return {'accuracy': accuracy_score(y_real, y_predicho), 'precision': precision_score(y_real, y_predicho, zero_division=0), 'sensibilidad': recall_score(y_real, y_predicho, zero_division=0), 'especificidad': especificidad, 'F1': f1_score(y_real, y_predicho, zero_division=0), 'AUC-ROC': roc_auc_score(y_real, probabilidad)}  # Devuelvo los indicadores.
metricas_train = calcular_metricas(y_train, pred_train, prob_train)  # Resumo train.
metricas_test = calcular_metricas(y_test, pred_test, prob_test)  # Resumo test.
comparacion = pd.DataFrame({'train (%)': {k: 100 * v for k, v in metricas_train.items()}, 'test (%)': {k: 100 * v for k, v in metricas_test.items()}})  # Organizo los porcentajes.
comparacion['diferencia test-train (p.p.)'] = comparacion['test (%)'] - comparacion['train (%)']  # Calculo diferencia en puntos porcentuales.
display(comparacion.round(1))  # Muestro métricas y brecha de generalización.
print(f"La diferencia train-test de AUC-ROC es {100 * (metricas_test['AUC-ROC'] - metricas_train['AUC-ROC']):+.1f} puntos; la de recall es {100 * (metricas_test['sensibilidad'] - metricas_train['sensibilidad']):+.1f} puntos.")  # Comento las brechas principales.

# Calculo la matriz de confusión y la curva ROC para test.
matriz_test = confusion_matrix(y_test, pred_test, labels=[0, 1])  # Cuento TN, FP, FN y TP en test.
fpr, tpr, umbrales = roc_curve(y_test, prob_test)  # Calculo tasas para dibujar ROC.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Creo dos paneles de evaluación.
ConfusionMatrixDisplay(matriz_test, display_labels=['DILI negativa', 'DILI positiva']).plot(ax=axes[0], cmap='Blues', colorbar=False)  # Muestro errores por clase.
axes[0].set_title('Matriz de confusión en test')  # Titulo el primer panel.
axes[1].plot(fpr, tpr, color='darkorange', label=f"AUC = {metricas_test['AUC-ROC']:.2f}")  # Dibujo la curva ROC.
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Clasificador aleatorio')  # Añado la referencia aleatoria.
axes[1].set_xlabel('Tasa de falsos positivos (1 - especificidad)')  # Etiqueto el eje horizontal.
axes[1].set_ylabel('Sensibilidad')  # Etiqueto el eje vertical.
axes[1].set_title('Curva ROC en test')  # Titulo el segundo panel.
axes[1].legend(loc='lower right')  # Identifico curva y referencia.
plt.tight_layout()  # Ajusto los paneles.
plt.show()  # Presento ambas evaluaciones juntas.
print(f"En test, especificidad = {metricas_test['especificidad']:.3f}: el modelo identifica correctamente esa proporción de negativos.")  # Interpreto la especificidad.
print(f"AUC-ROC = {metricas_test['AUC-ROC']:.3f}; valores cercanos a 0,5 indican poca capacidad de separar las dos clases.")  # Interpreto la discriminación.


## Interpretación de la evaluación

Las métricas de train y test permiten comprobar si el rendimiento se mantiene en datos no utilizados para ajustar el modelo. La sensibilidad informa sobre la proporción de positivos detectados y la especificidad sobre la de negativos correctamente identificados. La matriz de confusión permite localizar falsos positivos y falsos negativos, mientras que el AUC-ROC resume la capacidad de ordenar casos positivos por encima de negativos a distintos umbrales. La comparación debe considerar el desequilibrio de clases y la diferencia entre ambas particiones.